In [67]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.ensemble import RandomForestClassifier

import joblib

In [68]:
df = pd.read_csv("generated_data.csv").drop_duplicates()

# Clean column names
df.columns = df.columns.str.strip().str.upper()

# Replace empty strings with NaN
df.replace(r'^\s*$', np.nan, regex=True, inplace=True)

print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (253, 29)


,AGE,GENDER,DEPARTMENT,CHIEF_COMPLAINTS,COMORBIDITIES,RISKFACTORS,SURGICAL_HISTORY,SOCIAL_HISTORY,DIAGNOSIS,CLASSIFICATION_OF_UTI,...,WBC,POLYMORPHS,CRP,RFT_SERUM_CREATININE,SERUM_URIC_ACID,BLOOD_UREA,CUE_PUS_CELLS,EPITHELIAL_CELLS,PROTEINS,RBC
0,34,Female,General Medicine,Dysuria;Increased frequency,NaN,Poor hygiene,NaN,Non smoker,Acute cystitis,Uncomplicated,...,9800,68,6,0.9,4.8,28,12,4,Trace,3.0
1,52,Male,Urology,Fever;Flank pain;Dysuria,Diabetes,Catheterization,NaN,Non smoker,Acute pyelonephritis,Complicated,...,18200,78,65,2.1,6.2,48,28,6,Positive,6.0
2,27,Female,General Medicine,Dysuria;Urgency,NaN,Sexual activity,NaN,Non smoker,Acute cystitis,Uncomplicated,...,8700,60,4,0.8,4.1,24,10,5,Negative,2.0
3,68,Male,ICU,Fever;Flank pain;Vomiting,Diabetes;Hypertension,Catheterization,Prostate surgery,Ex smoker,Severe pyelonephritis,Complicated,...,22400,82,98,3.0,7.1,66,35,8,Positive,10.0
4,45,Female,General Medicine,Dysuria;Lower abdominal pain,Diabetes,NaN,Hysterectomy,Non smoker,Acute cystitis,Complicated,...,11200,70,18,1.1,5.0,32,18,6,Trace,4.0


In [69]:
numeric_features = [
    'AGE', 'CBP_LYMPHOCYTES', 'WBC', 'POLYMORPHS',
    'CRP', 'RFT_SERUM_CREATININE', 'SERUM_URIC_ACID',
    'BLOOD_UREA', 'CUE_PUS_CELLS', 'EPITHELIAL_CELLS', 'RBC'
]

# Force numeric conversion
for col in numeric_features:
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [70]:
df['TYPE_OF_BACTERIA'] = df['TYPE_OF_BACTERIA'].str.strip().str.lower()

df['TARGET'] = df['TYPE_OF_BACTERIA'].map({
    'gram negative': 0,
    'gram positive': 1
})

print(df['TARGET'].value_counts())

TARGET
0    177
1     76
Name: count, dtype: int64


In [71]:
categorical_features = [
    'GENDER', 'DEPARTMENT',
    'CLASSIFICATION_OF_UTI', 'TYPE_OF_UTI',
    'SITE_OF_INFECTION', 'TYPE_OF_SAMPLE',
    'PROTEINS'
]

In [72]:
text_features = [
    'CHIEF_COMPLAINTS', 'COMORBIDITIES',
    'RISKFACTORS', 'SURGICAL_HISTORY',
    'SOCIAL_HISTORY', 'DIAGNOSIS'
]

In [73]:
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

In [74]:
categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

In [80]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import FunctionTransformer

text_transformers = []

for col in text_features:
    
    text_pipeline = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value='')),
        
        # Convert DataFrame column to 1D array
        ('to_string', FunctionTransformer(
            lambda x: x.squeeze(), validate=False
        )),
        
        ('tfidf', TfidfVectorizer(
            lowercase=True,
            stop_words='english',
            ngram_range=(1,2),
            min_df=2,
            max_df=0.9
        ))
    ])
    
    text_transformers.append((col, text_pipeline, [col]))

In [81]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, numeric_features),
        ('cat', categorical_pipeline, categorical_features),
        *text_transformers
    ],
    remainder='drop'
)

In [82]:
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        random_state=42
    ))
])

In [83]:
feature_columns = numeric_features + categorical_features + text_features

X = df[feature_columns]
y = df['TARGET']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (202, 24)
Test shape: (51, 24)


In [84]:
model_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['AGE', 'CBP_LYMPHOCYTES',
                                                   'WBC', 'POLYMORPHS', 'CRP',
                                                   'RFT_SERUM_CREATININE',
                                                   'SERUM_URIC_ACID',
                                                   'BLOOD_UREA',
                                                   'CUE_PUS_CELLS',
                                                   'EPITHELIAL_CELLS', 'RBC']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleI...
                                                  ['SOCIAL_HISTORY']),
                                                 ('DIAGNOSIS',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(fill_value='',
                                                                                 strategy='constant')),
                                                                  ('to_string',
                                                                   FunctionTransformer(func=<function <lambda> at 0x0000014ED37E4040>)),
                                                                  ('tfidf',
                                                                   TfidfVectorizer(max_df=0.9,
                                                                                   min_df=2,
                                                                                   ngram_range=(1,
                                                                                                2),
                                                                                   stop_words='english'))]),
                                                  ['DIAGNOSIS'])])),
                ('classifier',
                 RandomForestClassifier(n_estimators=300, random_state=42))])

In [85]:
y_pred = model_pipeline.predict(X_test)
y_proba = model_pipeline.predict_proba(X_test)[:, 1]

print("Classification Report")
print(classification_report(y_test, y_pred))

print("ROC-AUC Score:", roc_auc_score(y_test, y_proba))

Classification Report
              precision    recall  f1-score   support

           0       1.00      0.97      0.99        36
           1       0.94      1.00      0.97        15

    accuracy                           0.98        51
   macro avg       0.97      0.99      0.98        51
weighted avg       0.98      0.98      0.98        51

ROC-AUC Score: 0.9935185185185185


In [86]:
accuracy = model_pipeline.score(X_test, y_test)
print("Test Accuracy:", accuracy)

Test Accuracy: 0.9803921568627451


In [ ]:
joblib.dump(model_pipeline, "model_1_bacteria.pkl")
print("Model saved successfully.")

Model saved successfully.


In [ ]:
X_test

,AGE,CBP_LYMPHOCYTES,WBC,POLYMORPHS,CRP,RFT_SERUM_CREATININE,SERUM_URIC_ACID,BLOOD_UREA,CUE_PUS_CELLS,EPITHELIAL_CELLS,...,TYPE_OF_UTI,SITE_OF_INFECTION,TYPE_OF_SAMPLE,PROTEINS,CHIEF_COMPLAINTS,COMORBIDITIES,RISKFACTORS,SURGICAL_HISTORY,SOCIAL_HISTORY,DIAGNOSIS
29,65,23,17600,77,58,2.4,6.6,54,27,8.0,...,Acute,Upper UTI,Urine,Positive,Fever;Flank pain,Diabetes,NaN,Prostate surgery,Non smoker,Complicated UTI
231,66,24,15600,73,42,1.9,7.1,54,26,4.0,...,Chronic,Lower,Catheter Urine,Positive,Frequency;Urgency,Hypertension,Catheterization,Prostate Surgery,Former Smoker,Catheter Associated UTI
71,70,21,23100,83,104,3.2,7.5,68,37,10.0,...,Acute,Upper UTI,Urine,Positive,Fever;Flank pain;Hypotension,Diabetes;CKD,Catheterization,Abdominal surgery,Ex smoker,Septic pyelonephritis
182,41,31,11600,69,21,1.2,5.7,36,17,6.0,...,Acute,Lower,Midstream Urine,Trace,Dysuria;Frequency,Diabetes,NaN,NaN,Non-smoker,Acute Cystitis
151,33,34,9200,63,5,0.9,4.7,24,14,7.0,...,Acute,Lower,Midstream Urine,Negative,Dysuria;Frequency,NaN,NaN,NaN,Non-smoker,Acute Cystitis
139,42,31,10900,67,16,1.1,5.5,33,15,8.0,...,Acute,Lower,Midstream Urine,Trace,Dysuria;Urgency,Hypertension,NaN,NaN,Non-smoker,Acute Cystitis
20,42,30,10900,69,20,1.1,5.1,30,17,6.0,...,Recurrent,Lower UTI,Urine,Trace,Dysuria;Lower abdominal pain,Diabetes,NaN,Hysterectomy,Non smoker,Recurrent cystitis
82,36,37,8300,60,5,0.8,4.3,24,6,NaN,...,Acute,Lower UTI,Urine,2,Dysuria;Frequency,NaN,Poor hygiene,NaN,Non smoker,Acute cystitis
239,54,23,18000,79,70,3.0,7.5,66,35,5.0,...,Acute,Upper,Midstream Urine,Positive,Fever;Flank Pain,Diabetes,Recurrent UTI,NaN,Smoker,Acute Pyelonephritis
234,35,34,9500,63,5,0.9,4.7,25,14,7.0,...,Acute,Lower,Midstream Urine,Negative,Dysuria;Frequency,NaN,NaN,NaN,Non-smoker,Acute Cystitis
